In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms

class StandardDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # 1. Auto-generate class dictionary (or define manually)
        # Sort ensures 0=Apple, 1=Banana consistently
        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_labels = {cls_name: i for i, cls_name in enumerate(classes)}

        self.image_paths = []
        self.labels = []

        # 2. Collect Paths
        for class_name, label in self.class_labels.items():
            paths = glob.glob(f"{root_dir}/{class_name}/*.*") # Matches .jpg, .png etc
            self.image_paths.extend(paths)
            self.labels.extend([label] * len(paths))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 3. Load & Process
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.RandomRotation(15)
])

train_dir = os.path.join(path, "PlantVillage", "test")
test_dir = os.path.join(path, "PlantVillage", "train")

train_dataset = StandardDataset(root_dir=train_dir, transform=transform)
test_dataset = StandardDataset(root_dir=test_dir, transform=transform)


# OR


from torchvision.datasets import ImageFolder

# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset  = ImageFolder(root=test_dir,  transform=transform)


In [ ]:
from PIL import Image
import os

# Load an image
image_path = os.path.join(path,"/kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/002a55fb-7a3d-4a3a-aca8-ce2d5ebc6925___RS_Early.B 8170.JPG")
image = Image.open(image_path)

# Display the image
image

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # it is better to run CNNs with GPUs for faster computation
device

In [ ]:
from IPython.display import clear_output
!pip install torch torchvision matplotlib scikit-learn
clear_output()

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
# Write your code here
class SimpleFashionCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(

            # TRACE THE SHAPE OF THE TENSORS AS IT PASSES THROUGH THE CONV LAYERS TO AVOID SHAPE MISMATCH ERRORS
            # HOW? --> by using the output features formula shown above

            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [B,32,28,28]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,32,14,14]

            # TO-DO: Add second Conv2d layer (input: 32, output: 64, kernel: 3, padding: 1)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # [B,64,14,14]
            nn.ReLU(),
            # TO-DO: Add MaxPool2d with kernel_size=2
            nn.MaxPool2d(2),                            # [B,64,7,7]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TO-DO: Calculate input features (channels × height × width)
            # After 2 MaxPool2d(2): 28 → 14 → 7
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # TO-DO: Pass x through features
        x = self.features(x)
        # TO-DO: Pass result through classifier
        x = self.classifier(x)
        return x



In [ ]:
model = SimpleFashionCNN().to(device)
model

In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    # TO-DO: Get predicted class indices
    # HINT: Use torch.argmax with dim=1
    preds = torch.argmax(logits, dim=1)
    # TO-DO: Calculate and return accuracy
    # HINT: Compare preds with labels, convert to float, mean, then .item()
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for batch in tqdm(loader):
        images, labels = batch['image'], batch['label']
        images, labels = images.to(device), labels.to(device)

        # TO-DO: Zero the gradients
        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        # TO-DO: Backward pass
        loss.backward()

        # TO-DO: Update parameters
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for batch in tqdm(loader):
            images, labels = batch['image'], batch['label']
            images, labels = images.to(device), labels.to(device)

            # TO-DO: Get model predictions
            logits = model(images)

            # TO-DO: Calculate loss
            # HINT: Use the criterion function
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
# Training setup
# TO-DO: Define loss criterion
# HINT: What loss function works for multi-class classification?
criterion = nn.CrossEntropyLoss()

# TO-DO: Set learning rate
learning_rate = 0.001  # Typical range: 0.0001 to 0.01

# TO-DO: Define optimizer
# HINT: Pass model.parameters() and learning_rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# TO-DO: Set number of epochs
num_epochs = 5  # How many times to iterate through the dataset?

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')



In [ ]:
# Write your code here
